In [0]:
from pyspark.sql.functions import col, explode, to_date, concat_ws

# 1. Read Bronze raw table
bronze_df = spark.read.table("f1_project.default.f1_project_data_bronze")

# 2. Explode Races array (Creates one row per Grand Prix)
races_df = bronze_df.select(
    explode("MRData.RaceTable.Races").alias("race"),
    col("ingestion_timestamp")
)

# 3. Explode Results array and flatten struct fields into typed columns
silver_df = races_df.select(
    col("race.season").cast("int").alias("season"),
    col("race.round").cast("int").alias("round_number"),
    col("race.raceName").alias("race_name"),
    to_date(col("race.date")).alias("race_date"),
    col("race.Circuit.circuitId").alias("circuit_id"),
    col("race.Circuit.circuitName").alias("circuit_name"),
    
    # Explode driver results array to get one row per driver
    explode(col("race.Results")).alias("result"),
    col("ingestion_timestamp")
).select(
    "season",
    "round_number",
    "race_date",
    "race_name",
    "circuit_id",
    "circuit_name",
    
    # Finishing & Race Metrics
    col("result.position").cast("int").alias("position"),
    col("result.positionText").alias("position_text"),
    col("result.points").cast("float").alias("points"),
    col("result.grid").cast("int").alias("grid_position"),
    col("result.laps").cast("int").alias("laps_completed"),
    col("result.status").alias("status"),
    
    # Driver Attributes
    col("result.Driver.driverId").alias("driver_id"),
    col("result.Driver.code").alias("driver_code"),
    col("result.Driver.permanentNumber").alias("driver_number"),
    concat_ws(" ", col("result.Driver.givenName"), col("result.Driver.familyName")).alias("driver_name"),
    col("result.Driver.nationality").alias("driver_nationality"),
    
    # Constructor Attributes
    col("result.Constructor.constructorId").alias("constructor_id"),
    col("result.Constructor.name").alias("constructor_name"),
    
    # Telemetry / Lap Highlights
    col("result.FastestLap.lap").cast("int").alias("fastest_lap_number"),
    col("result.FastestLap.Time.time").alias("fastest_lap_time"),
    
    "ingestion_timestamp"
)

# 4. Upsert (MERGE INTO) Target Silver Table
silver_df.createOrReplaceTempView("f1_project_data_silver")

# Ensure Target Table Exists
spark.sql("""
CREATE TABLE IF NOT EXISTS f1_project.default.f1_project_data_silver
USING DELTA AS SELECT * FROM f1_project_data_silver WHERE 1=0
""")

spark.sql("""
    MERGE INTO f1_project.default.f1_project_data_silver AS target
    USING f1_project_data_silver AS source
    ON target.season = source.season 
   AND target.round_number = source.round_number 
   AND target.driver_id = source.driver_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print("Silver Transformation Complete.")

In [0]:
%sql
select * from f1_project.default.f1_project_data_silver